# పాఠం 18: క్రిప్టోగ్రాఫిక్ రసీద్యులతో AI ఏజెంట్లను సురక్షితం చేయడం

## ప్రాక్టికల్ నోట్‌బుక్

ఈ నోట్‌బుక్ నాలుగు వ్యాయామాలను చూపిస్తుంది:

1. ఏజెంట్ టూల్ కాల్ కోసం **మీ మొదటి రసీదు సంతకం చేయండి** మరియు దాన్ని వ్యధారీతిగా నిర్ధారించండి.
2. **రసీదును చీ_DISTURB_ చేసి** నిర్ధారణ విఫలం అవడానికి చూడండి.
3. **మూడు రసీదు శ్రేణిని నిర్మించి** శ్రేణి సమగ్రతను నిర్ధారించండి.
4. **Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ టూల్ కాల్‌ను ర్యాప్ చేయండి** తద్వారా ప్రతి చర్య ఒక రసీదు విడుదల చేస్తుంది.

అన్ని క్రిప్టోగ్రాఫిక్ ప్రాథమికతలు బాగా నిర్వహించే లైబ్రరీల నుండి దిగుమతి చేయబడ్డాయి (`pynacl` Ed25519 కోసం, `jcs` RFC 8785 కానానికల్ JSON కోసం, Python స్టాండర్డ్ లైబ్రరీ నుండి `hashlib` SHA-256 కోసం). రసీదు లాజిక్ స్వయంగా సాదా Python కాబట్టి మీరు చూడగలరు మరియు మార్పులు చేయొచ్చు.

సెల్స్‌ను క్రమంగా నడపండి. ప్రతి విభాగం సంక్షిప్తంగా మరియు స్వీయసంస్థితంగా ఉంటుంది.


## సెటప్

రెండు డిపెండెన్సీలను ఇన్స్టాల్ చేయండి. వాటి ఇద్దరికీ అనుమతిపూర్వక లైసెన్సులు (Apache-2.0 / MIT) ఉన్నాయి.


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## సహాయక ఉపకరణాలు

ఈ రెండు సహాయకులు base64url ఇన్‌కోడింగ్ (padding లేకుండా) మరియు ఏదైనా వస్తువుల SHA-256 హ్యాషింగ్‌ని నిర్వహిస్తాయి. వారు నోట్బుక్ మిగిలిన భాగాన్ని రసీదు లాజిక్ మీదే కేంద్రీకరించడంలో సహాయపడతారు.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## సెక్షన్ 1: మీ మొదటి రసీదు సైన్ చేయండి

మన ఏజెంట్ **Contoso Travel** వారి కస్టమర్ కోసం సిడ్నీ నుండి లాస్ ఏంజిల్స్ వరకు ఫ్లైట్లను చూసినట్టు ఊహించుకుందాం. ఈ టూల్ కాల్ గురించి గేట్‌వే యొక్క ప్రకటనను రికార్డ్ చేయాలి, తద్వారా భవిష్యత్తులో ఒక ఆడిటర్ అది ప్రత్యేకంగా అటాచ్ చేసిన జారీదారు కీ క్రింద పరిశీలించగలడు.

### దశ 1.1: సైన్ చేయడానికి ఓ కీ రూపొందించండి

ప్రొడక్షన్‌లో, ఏజెంటിന്റെ సైన్ చేయు కీ హార్డ్‌వేర్ సెక్యూరిటీ మాడ్యూల్ (HSM), Azure కీ వాల్ట్, లేదా ఇలాంటి ఒక రక్షిత నిల్వలో ఉంటుంది. ఈ పాఠంలో మనం మెమరీలో ఒక కొత్త కీని ఉత్పత్తి చేస్తున్నాము.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### దశ 1.2: రసీదు పేలోడ్‌ని రూపొందించండి

పేలోడ్‌లో రసీదు ధృవీకరించాల్సిన అన్ని సమాచారముంటుంది: ఎవరు చర్య తీసుకున్నారు, ఏ సాధనం, ఎలాంటి ఆర్గ్యుమెంట్లతో, ఏమి తిరిగి వచ్చింది, ఏ విధానంపై, మరియు ఎప్పుడు. రసీదు సున్నితమైన విషయాలను లీక్ చేయకూడదనే కారణంగా ఆర్గ్యుమెంట్లు మరియు ఫలితాన్ని పంక్తిలో చేర్చకుండా హ్యాష్ చేస్తాం.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### దశ 1.3: పత్రాన్ని సంతకం చేసి సంగ్రహించండి

మూడు దశలు:

1. JCS ఉపయోగించి payloadని canonicalize చేయండి తద్వారా రెండు అమలులను ఒకే తార్కిక పత్రం సిద్ధం చేస్తే బైట్స్ నిర్దોષంగా జరుగుతాయి.
2. Ed25519 ప్రైవేట్ కీతో నేరుగా canonical bytesను సంతక చేయండి. PureEdDSA లో సందేశం అంతర్గతంగా హాష్ అవుతుంది, కాబట్టి అదనపు pre-hash ప్రయోగం ప్రోటోకాల్ మార్చుతుంది.

ఆ సంతకం ఆ తర్వాత అసలు payloadకు జోడించి తుది పత్రం తయారవుతుంది.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### దశ 1.4: రసీదును ధృవీకరించడం  

ధృవీకరణ ప్రక్రియను తిప్పి చేస్తుంది. మేము సంతకం తీసివేసి, కానోనికల్ బైట్లను పునః గణన చేసి, సంతకాన్ని ఇష్యు చేసిన వారి పబ్లిక్ కీతో సరిపోల్చి చూద్దాము.  

ధృవీకరణ ఆఫ్లైన్‌లో జరుగుతుంది: కాల్ చేసుకునే సర్వీస్ లేదా నెట్‌వర్క్ ప్రాప్యత లేదు. రసీదుతో పాటు ఇంకొక విషయం అవసరం: ఇష్యు చేసిన వారి పబ్లిక్ కీ, అది ఆడిటర్ ఇప్పటికే నమ్మిన ఛానెల్ ద్వారా పొందాలి, ఉదాహరణకు ఇష్యూ చేసిన వారు ప్రచురించే JWK సెట్ లేదా ముందుగానే ఫిక్స్ చేయబడిన విలువ. రసీదు లోపల ఉన్న `public_key` ఫీల్డ్ ఆ పాత్రను పోషించలేని కారణం, రసీదు రాసే వారు ఆ ఫీల్డ్ కూడా రాస్తారు. విభాగం 2 ఈ వఛిత తయారీని నిరోధించే విధంగా చూపిస్తుంది.  


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


మీరు `Receipt is valid: True`, `Pre-hashed receipt valid: False` మరియు `Malformed receipts rejected: True` ను చూడాలి. పాజిటివ్ కేసు డైరెక్ట్-JCS సంతకం మార్గం పనిచేస్తుందని సిద్దాంతం చేస్తుంది; నెగటివ్ కంట్రోల్ సంతకం-స్కోపు నియమాన్ని ప్రవహనీయమైనదిగా చేస్తుంది, ప్రాసుగా వదిలేయడానికి బదులుగా; చివరి లైన్ తప్పు ఆకృతిలో ఉన్న ఇన్‌పుట్‌ను కోడను తగులుతుండటానికి బదులుగా `False` ను తిరిగి ఇవ్వడాన్ని చూపుతుంది.


## విభాగం 2: రసీదు లో మార్పులు చేయడం

రసీదుల ఉద్దేశ్యం అవి మార్పును స్పష్టం చేసే విధంగా ఉండటం. దీన్ని సాబితీ చేద్దాం.

రసీదులో కేవలం ఒక అక్షరాన్ని మార్చి ధృవీకరణ విఫలమవుతుందో చూడుకుందాం.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### ఏమయ్యింది?

మనము `policy_id` మార్చినప్పుడు, కానానికల్ బైట్స్ మారాయి. సంతకం (ఆదికాల కానానికల్ బైట్స్ మీద ఉన్నది) πλέον సరిపోదు. నిర్ధారణ సంధానం సరిగ్గా `False` ను తిరిగి ఇస్తుంది.

సంతకం చేసిన ఫీల్డ్‌ని మార్చడం సాధ్యం లేదు, అంటే `signature` ఆబ్జెక్టు వెలుపల ఏదైనా మార్పు చేసి, వ్యవస్థాపకుడి పబ్లిక్ కీతో రసీదును ధృవీకరించలేము, తప్ప దాడిదారుడు వ్యవస్థాపకుడి ప్రైవేట్ కీ కలిగి ఉంటే. `signature` ఆబ్జెక్టు స్వయంగా సంతకం చేయబడలేదు: `sig` అదే సంతకం, మరియు `verify_receipt` `public_key` ని పట్టించుకోదు, కాబట్టి ఆ ఫీల్డ్ ఎడిట్ చేయడం వల్ల ఏమీ మారదు. ప్రైవేట్ కీ కీ వాల్ట్ లో ఉంటే మరియు ధృవీకరణదారులు వ్యవస్థాపకుడి స్వతంత్రంగా ప్రచురించిన కీ ని ఉపయోగించే వరకు, సంతకం ఉన్న ఫీల్డ్ లో మార్పు దాచిపెట్టడం అసాధ్యం. ఆ పరిస్థితి ముఖ్యం: తరువాతి సేలు వేరు కీ తో సరిగ్గా ధృవీకరించగల ఒక మార్చబడిన రసీదును చూపిస్తుంది.

మీరు అనుసరించండి: పై సేలు లో `tool_name` లేదా `agent_id` లేదా `timestamp` ను మార్చి మళ్లీ అమలు చేయండి. సంతకం చేసిన ఫీల్డ్ లో ప్రతి మార్పు అవ్యవస్థిత రసీదును ఉత్పత్తి చేస్తుంది.


### సంతకం పట్టుకోని దుర్వినియోగం

పై దుండగుడు తిరిగి సంతకం చేయలేదు. మరింత జాగ్రత్తగా ఉన్నవాడు చేస్తాడు: రెసీట్ను మార్చి,
తాము స్వయంగా రూపొందించిన కీతో సంతకం చేసి, ఆ కీని
`public_key` ఫీల్డ్లో ఉంచుతారు. ఫలితం ఏమిటంటే రెసీట్లో సంతకం ఆ విషయం కోసం
పూర్తిగా నిజమైనది.

దీన్ని నడిపించి, ఏ రెండు పరీక్షలలో అది పట్టుకుంటుందో చూడండి.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


మీరు `True` తరువాత `False` చూడాలి. ఆ మూడు `assert` లైన్లు దీన్ని ఒక
తనిఖీగా చేస్తాయి: ఫార్జరీ ఎప్పుడైనా
జారీదారు యొక్క కీకి సరిపోతే సెల్ ఒక పొరపాటు చూపిస్తుంది.

ఫార్జరీ అంతర్గతంగా అనుకూలంగా ఉంటుంది. దాని సంతకం నిజంగానే దాని
పాఠ్యాంశాలు మరియు దాని పరిధిలో ఉన్న కీకి సరిపోతుంది, కాబట్టి కీని రసీదులో నుండి చదివిన ధృవీక‌ర‌ణ రిపోర్ట్‌ను ఇది చెల్లుబాటు అయ్యింది అని తెలియజేస్తుంది. రసీదు గురించి ఏమీ తప్పు కనిపించ‌దు, ఎందుకంటే
దాని గురించి ఏమీ అస‌మంజ‌స‌మే లేదు.


దాన్ని ఎ捕ిక చేసే ఒక్కటి ఏమిటంటే దాడిదారు ప్రభావితం చేయలేని కీతో తనిఖీ చేయడం మాత్రమే.
అందుకే `verify_receipt` కీని ఆర్గ్యుమెంట్‌గా తీసుకుంటుంది: ఒక
కోడ్ సైన్‌కు కలిగిన ఫంక్షన్ `verify_receipt(receipt)` సురక్షితంగా రాయబడలేదు,
ఎందుకంటే దాని చేరుకునే ఏకైక కీ దాడిదారు అందించినదే.

అదే రసీదు సంపాదించబడుతుంది `sample_receipts/04_resigned_receipt.json` గా మీరు
నోట్లు బహిర్గతం నుండి దాన్ని పరిశీలించాలనుకుంటే.


## భాగం 3: రసీదు లాంటి చైన్‌లను కట్టండి

ఒక రసీదు ఒక చర్యను రక్షిస్తుంది. ఎక్కువమంది ఏజెంట్స్ అనేక చర్యలు తీసుకుంటారు. సమగ్ర శ్రేణిని టాంపర్-ఎవీడెంట్ గా చేయడానికి, ప్రతి కొత్త రసీదులో పూర్వపు రసీదు యొక్క హ్యాష్‌ను చేర్చడం ద్వారా ప్రతీ రసీదును పూర్వపు రసీదుతో లింక్ చేస్తాము.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ఎవరైనా అందించిన శ్రేణి లో రసీదును తొలగించగలిగితే లేదా పునఃవిన్యస్తం చేస్తే, తదుపరి లింక్ బ్రేక్ అవుతుంది. ట్రంకేషన్ లేదా సైన్‌ చేయబడిన కంట్రోల్ suffix రీవ్రైటింగ్‌ను గుర్తించడానికి ప్రస్తుతం పిన్ చేసిన హెడ్లు, బాహ్య చెక్పాయింట్ లేదా ఆటెంటికేటెడ్ పాపులేషన్ కమిట్‌మెంట్ అవసరం.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


ఇప్పుడు మధ్య రసీదుతో మిడిపట్టి రసం చెయండి మరియు మళ్లీ ధృవీకరించండి. మిడిపట్టిన రసీదు దాని సంతకం తనిఖీ విఫలమవుతుంది, మరియు తదుపరి రసీదు అనుబంధం తనిఖీ విఫలమవుతుంది (ఎందుకంటే దాని `previous_receipt_hash` ఇప్పుడు మార్చబడిన మధ్య రసీదుతో సరిపోలడం లేదు).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


రసీది 0 ఇంకా సరైనదిగా ఉంది (దాన్ని మార్చలేదు మరియు దానికి ఆధారపడే మరేదైనా ముందస్తు రసీదు లేదు). రసీది 1 సంతకం తనిఖీలో వైఫల్యం అవుతుంది ఎందుకంటే మేము `tool_args_hash` మార్చాము. రసీది 2 తన చైన్-లింక్ తనిఖీలో విఫలం అవుతుంది ఎందుకంటే దాని `previous_receipt_hash` అసలు (ఇప్పుడు మార్చబడిన) రసీదు 1 పై ఆధారపడి లెక్కించబడింది.

ఒక దుండగుడు మార్చబడిన రసీదు 1 ను మళ్లీ సంతకం చేసినా, వారు జారీదారుడి ప్రైవేట్ కీ కలిగి లేకపోతే, కొత్త సంతకం జారీదారుడి కీ కంటే విఫలమవుతుంది (వారి స్వంత కీతో మళ్లీ సంతకం చేయడం విఫలం అవుతుంది, సెక్షన్ 2 లో చూపించినట్లు), మరియు రసీదు 2 లో చైన్-లింక్ అసమ్మత భేరతను ఇంకా చూపిస్తుంది. మార్పును దాచడానికి, దుండగుడు మార్చడం ప్రారంభమైన బిందువు నుండి ప్రతీ రసీదును మళ్లీ సంతకం చేయాల్సి ఉంటుంది, దీనికీ జారీదారుడి ప్రైవేట్ కీ అవసరం.


## విభాగం 4: ఏజెంట్ టూల్ కాల్‌ను రసీదు సంతకం చేయడం తో చుట్టుముట్టు

నిజమైన అమలులో, మీరు ప్రతి ఏజెంట్ రచయితకు `make_receipt` పిలవడం గుర్తుంచుకోవాలని అనుకోరు. ప్రతి టూల్ పిలుపుల కోసం రసీదు సంతకం ఆటోమేటిక్ కావాలని మీరు కోరుకుంటారు.

ఇక్కడ సర్వసాధారణమైన నమూనా ఉంది: ఏదైనా పిలవగల టూల్ ఫంక్షన్ తీసుకుని దాని నుండి రసీదు జారీ చేసే వెర్షన్‌ను తిరిగి ఇస్తుంది.Wraper క్లాస్. ఇది Microsoft Agent Framework (`agent_framework.foundry`) సహా ఏ ఏజెంట్ ఫ్రేమ్‌వర్క్‌కు అనుగుణంగా ఉంటుంది.

మీ దగ్గర Microsoft Foundry ప్రాజెక్టు ఏర్పాటుచేయబడకపోయినా, క్రింది స్థానిక మాక్ నమూనాను చూపిస్తుంది.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft Agent Framework తో సమన్వయం చేయడం

పై `ReceiptedTool` ర్యాపర్ ఫ్రేమ్‌వర్క్-నీయమైనది కాదు. Microsoft Agent Framework తో నిర్మించిన ఏజెంట్ లో దీన్ని ఉపయోగించడానికి, ర్యాప్ చేసిన ఫంక్షన్‌ను టూల్‌గా నమోదు చేయండి. ఒక స్నాప్‌షాట్ (మీరు మాక్‌ని నిజమైన Microsoft Foundry టూల్ నమోదు తో మార్చుతారు):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ఏజెంట్ ఫ్రేమ్‌వర్క్‌కు రసీట్లు ఎలాంటి సమాచారం అవసరం ఉండదు. రసీదు సంతకం టూల్ చుట్టూ ఉంది, ఫ్రేమ్‌వర్క్‌లో అనుసంధానించబడలేదు. ఇది ఏజెంట్ కోడ్‌ను తిరిగి రాయకుండా ప్రావెనెన్స్ ను జోడించే విధానం. 


## సారాంశం మరియు వాయిస్తుంది సవాలు

మీరు:

- Ed25519 కీ జంటను రూపొందించారు.
- ఏజెంట్ టూల్ కాల్ కోసం రసీదు నిర్మించి సంతకం చేశారు.
- రసీదును ఆఫ్‌లైన్‌లో జారీదారు యొక్క పబ్లిక్ కీని మాత్రమే వినియోగించి, రసీదు బయటపడిన తరువాత ధృవీకరించారు.
- రసీదుతో చట్టపరమైన మార్పు చేసి ధృవీకరణ విఫలమైనది చూశారు.
- మూడు రసీదుల యొక్క హాష్-చైన్ సీక్వెన్స్ సేకరించారు.
- చైన్ మధ్యలో మార్పు చేయడంతో సంతకం విఫలం మరియు చైన్-లింక్ విఫలమయ్యాయి చూశారు.
- ఆటోమేటిక్ రసీదు సంతకం తో ఒక టూల్ ఫంక్షన్‌ను చుట్టుకున్నారు.

**వాయిస్తుంది సవాలు.** రసీదు స్కీమాలో `request_id` ఫీల్డ్‌ను (విభజన టేసింగ్ కోసం UUID) విస్తరించండి. దానిని కలిగి ఉండడానికి `make_receipt` ను అప్‌డేట్ చేసి, రసీదులు అంతటా ధృవీకరిస్తున్నాయనా అనేది ధృవీకరించండి. తరువాత సంతకం తర్వాత ఫీల్డ్‌ను మార్చి ధృవీకరణ విఫలమవుతుందో కూడా పరిశీలించండి. ఇది ప్రతి బైట్ యొక్క కెనానికల్ ఎంకోడింగ్ సంతకానికి ఎలా సహాయం చేస్తుందో మీరు అవగాహన పొందగలుగుతారు.

**ముఖ్యమైన సరిహద్దు.** ఒక పిన్న కీ మరియు చైన్ సందర్భంలో, రసీదులు సంతకం ఆతtributions, కంటెంట్ సమగ్రత, మరియు ప్రదర్శిత ముందస్తు లింక్ లను నిరూపిస్తాయి. అవి అమలు, మూల సత్యం, నమ్మకమైన సమయం, పూర్తి స్థితి, విధాన మూల్యాంకనం లేదా సరిగా ఉనట్టును నిరూపించవు. రసీదులు ఒక ప్రాతిపదిక. గవర్నెన్స్ మీరు పైగా నిర్మించే వ్యవస్థ.

ఆ సరిహద్దును దృష్టిలో పెట్టుకుని పాఠం READMEని మళ్ళీ చదవండి. టీమ్స్ పేదాటులో సాధారణ తప్పు "మాకు రసీదులు ఉన్నాయి" అంటే "మేము గవర్నెన్స్ లో ఉన్నాం" అని అర్థం చేసుకోవడమే. అలాంటిది కాదు. రసీదులు ఏజెంట్ ప్రవర్తనను ఆడిట్ చేయదగినది చేసేవి. అవి దాన్ని సరి చేసేవి కావు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
